# Advanced Analytics for a Better World — Lecture 2\n\n[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/aabw/notebooks/lecture-2/shortest-path-optimization-vs-algorithms.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/aabw/notebooks/lecture-2/shortest-path-optimization-vs-algorithms.ipynb)\n\n> Original Lecture 2 notebook, retained with its author attribution and content.\n

# Shortest path as part of Linear Optimization and as part of Computer Science

Joaquim Gromicho, 2026

The shortest path problem is a classical example that sits at the intersection of Linear Optimization and Computer Science.

From the perspective of Linear Optimization, shortest path problems can be formulated naturally as linear programs. A standard formulation uses flow conservation constraints together with arc variables indicating whether an edge is used. Although the variables are naturally integer, the constraint matrix of the problem is totally unimodular. As a consequence, the linear programming relaxation already yields integer solutions. In other words, the problem is inherently discrete but does not require explicitly forcing the variables to be integer.

This illustrates an important strength of linear optimization. Many graph problems can be expressed as linear models in a very natural way. Linear optimization therefore provides a unifying modelling framework capable of representing a wide range of combinatorial problems, including network flows, assignment, matching, and shortest paths.

However, while the linear programming formulation is elegant and conceptually powerful, in practice shortest path problems are almost always solved using specialized algorithms. Dedicated algorithms exploit the structure of graphs much more directly and are typically far more efficient than solving a general purpose linear program.

The most famous of these algorithms is Dijkstra's algorithm, introduced by Edsger W. Dijkstra in 1959. Dijkstra was a computer scientist, and his work exemplifies the algorithmic tradition within Computer Science that focuses on designing efficient procedures for specific problem classes.

It is also important to note that an algorithm is not just an abstract idea, its performance depends strongly on the way it is implemented and on the data structures that support it. For example, different priority queue implementations lead to different theoretical and practical running times. Binary heaps, Fibonacci heaps, and other structures change the complexity bounds and the empirical performance of Dijkstra's algorithm.

In summary, the shortest path problem highlights a productive interaction between Linear Optimization and Computer Science. Linear optimization provides a clean mathematical formulation and theoretical insight, while algorithmic approaches provide highly efficient computational methods tailored to the structure of the problem.

In [ ]:
import sys
at_colab = "google.colab" in sys.modules

In [ ]:
if at_colab:
    import os
    %pip install idaes-pse --pre >/dev/null 2>/dev/null
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

In [ ]:
import pyomo.environ as pyo
import networkx as nx
import pandas as pd
import numpy as np
from io import StringIO
import matplotlib.pyplot as plt
from time import perf_counter as pc

In [ ]:
nodesFromLecture = '''
node;x;y
A;0;1
B;2;2
C;5;2
D;2;0
E;5;0
'''

edgesFromLecture = '''
from;to;length
A;B;10
A;D; 5
B;C; 1
B;D; 2
C;E; 4
D;B; 3
D;C; 9
D;E; 2
E;A; 7
E;C; 6
'''

edges = pd.read_csv(StringIO(edgesFromLecture), sep=";",index_col=['from','to'])
nodes = pd.read_csv(StringIO(nodesFromLecture), sep=";",index_col='node')

In [ ]:
nodes

In [ ]:
edges

In [ ]:
def FillGraphFromFrames( g, nodes, edges ):
    """ Fills a graph with nodes and edges and the corresponding attributes from data frames.
    The nodes frame is expected to be indexed on the node names and the edges frame on the tuples of nodes defining  the edges.
    Args:
        g (graph): either a nx.Graph or a nx.DiGraph object
        nodes (dataframe): one node per row, the columns define attribute values
        edges (dataframe): one edge per row, the columns define attribute values

    Returns:
        graph: the graph g taken as input extended with the nodes and edges from the dataframes
    """
    for node, row in nodes.iterrows():
        g.add_node(node,**row.to_dict())
    for edge, row in edges.iterrows():
        g.add_edge(*edge,**row.to_dict())
    return g

In [ ]:
def draw_graph(G: nx.Graph, solution : list = [ ]) -> None:
    """
    Draws a networkx graph using the specified node positions, edge lengths, and edge colors.

    :param G: A networkx Graph object with nodes that have 'x' and 'y' attributes for position
              and edges that have 'length' and 'color' attributes.
    """

    # Extract node positions from the node attributes
    pos = {node: (data['x'], data['y']) for node, data in G.nodes(data=True)}

    # Draw the graph using the node positions
    nx.draw_networkx_nodes(G, pos, node_size=500)
    nx.draw_networkx_labels(G, pos, font_color='lightblue', font_size=13)

    # Draw the graph edges with the specified colors and add edge labels with the length
    edge_colors = ['red' if (s,t) in solution else 'black' for s,t in G.edges]
    edge_labels = nx.get_edge_attributes(G, 'length')
    nx.draw_networkx_edges(G, pos, edge_color=edge_colors)
    nx.draw_networkx_edge_labels(G, pos, label_pos=.2, edge_labels=edge_labels)

    # Show the plot
    plt.show()

In [ ]:
G = FillGraphFromFrames( nx.DiGraph(), nodes, edges )

In [ ]:
draw_graph(G)

In [ ]:
G.nodes(data=True)

In [ ]:
G.edges(data=True)

In [ ]:
def ShortestPathAsLinearOptimization(G,s,t,attribute='length'):

    G_succ  = G._succ if G.is_directed() else G._adj
    G_pred  = G._pred if G.is_directed() else G._adj

    edges = list(G.edges())
    if not G.is_directed():
        edges = edges + [(j,i) for i,j in edges]

    m = pyo.ConcreteModel('sp')

    m.N = pyo.Set( initialize=list(G.nodes()) )
    m.E = pyo.Set( initialize=list(edges) )
    m.x = pyo.Var( m.E, within=pyo.NonNegativeReals )

    @m.Param(m.E)
    def c(m,i,j):
        return G.get_edge_data(i,j).get(attribute,0)

    @m.Param(m.N)
    def b(m,j):
        if j == s:
            return 1
        elif j == t:
            return -1
        return 0

    @m.Objective( sense=pyo.minimize )
    def length(m):
        return pyo.quicksum( m.c[e]*m.x[e] for e in m.E )

    @m.Constraint(m.N)
    def balance(m,j):
        return pyo.quicksum( m.x[(j,k)] for k in G_succ[j].keys() ) - \
               pyo.quicksum( m.x[(i,j)] for i in G_pred[j].keys() ) == m.b[j]

    return m

In [ ]:
m = ShortestPathAsLinearOptimization(G,'A','C')
pyo.SolverFactory('cbc').solve(m)
m.pprint()

In [ ]:
def SolveAsLO( G, s, t, solver='cbc' ):
  m = ShortestPathAsLinearOptimization(G,s,t)
  pyo.SolverFactory(solver).solve(m)
  return [ e for e,v in m.x.items() if v() > .5 ], m.length()

In [ ]:
sol, length = SolveAsLO( G, 'A', 'C' )
sol, length

In [ ]:
draw_graph( G, sol )

In [ ]:
def SimpleDijkstra( G, start, terminus, attribute='length' ):

    def DijkstraInitialize(start):
        labels = dict()
        labels[start] = 0
        return start,{start},labels,dict()

    def DijkstraScan(G,current,labels,prev,attribute):
        for _,suc,length in G.edges(current,data=attribute):
            if labels[current]+length < labels.get(suc,np.inf):
                labels[suc] = labels[current]+length
                prev[suc] = current
        return labels,prev

    def DijkstraAdvance(labels,permanent):
        return min(labels.keys()-permanent,key=labels.get)

    def DijkstraBacktrack(current,prev,start):
        path = [current]
        while current != start:
            current = prev[current]
            path.append(current)
        path.reverse()
        return path

    current,permanent,labels,prev = DijkstraInitialize(start)
    while current != terminus:
        labels,prev = DijkstraScan(G,current,labels,prev,attribute)
        current = DijkstraAdvance(labels,permanent)
        permanent.add(current)
    return DijkstraBacktrack(current,prev,start),labels[terminus]


In [ ]:
SimpleDijkstra( G, 'A', 'C' )

In [ ]:
def Dijkstra( G, start, terminus, attribute='length' ):

    from heapq import heappush as push, heappop as pop

    G_succ  = G._succ if G.is_directed() else G._adj
    G_pred  = G._pred if G.is_directed() else G._adj

    current = start
    dist    = { current : 0 }
    labels  = []
    while current != terminus:
        for succ, e in G_succ[current].items():
            if succ not in dist:
                push(labels,(dist[current]+e[attribute],succ))
        (d, current) = pop(labels)
        while current in dist:
            (d, current) = pop(labels)
        dist[current] = d
    path = [current]
    while current != start:
        for pred, e in G_pred[current].items():
            if pred in dist:
                if dist[pred]+e[attribute] == dist[current]:
                    current = pred
                    path.append(current)
                    break
    return path[::-1],dist[terminus]


In [ ]:
Dijkstra( G, 'A', 'C' )

In [ ]:
def GenerateGraph( n, seed=2022 ):
  g = nx.connected_watts_strogatz_graph(n, 10, .8, seed=seed)
  l = { e : r for e,r in zip(g.edges(),np.random.randint(1,10,len(g.edges()))) }
  nx.set_edge_attributes(g, l, "length")
  return g

In [ ]:
strategies= [ SolveAsLO, SimpleDijkstra, Dijkstra ]

In [ ]:
def DoThese( n_max, strategies, step=10 ):
  timings = pd.DataFrame( index = range(10,n_max,step), columns=[ f.__name__ for f in strategies ] )
  values = timings.copy()
  for n in timings.index:
    g = GenerateGraph(n)
    for f in strategies:
      t = pc()
      p,v = f(g,0,n-1)
      t = pc()-t
      timings.at[n,f.__name__] = t
      values.at[n,f.__name__] = v
  return timings, values

In [ ]:
DoThese( 100, [ SolveAsLO, SimpleDijkstra, Dijkstra ] )[0].plot()

In [ ]:
DoThese( 1000, [ SimpleDijkstra, Dijkstra ], step=100 )[0].plot()

In [ ]:
if at_colab:
  %pip install line_profiler

In [ ]:
%load_ext line_profiler

In [ ]:
g = GenerateGraph(100000)

In [ ]:
%lprun -u 1e-3 -f SimpleDijkstra SimpleDijkstra(g,0,len(g.nodes())-1)

In [ ]:
%lprun -u 1e-3 -f Dijkstra Dijkstra(g,0,len(g.nodes())-1)

In [ ]:
def nx_sp(g,s,t):
  return nx.shortest_path(g,s,t), nx.shortest_path_length(g,s,t)

In [ ]:
DoThese( 1000, [ Dijkstra, nx_sp ], step=100 )[0].plot()